# Destilação vs. treino direto — pipeline passo a passo

Cada célula chama as funções de `src/` com os valores de `src/config.py`. Tudo é cacheado:
rodar uma célula de novo pula o que já terminou, e um treino interrompido retoma da última época.

In [ ]:
SIZE = "S"          # aluno usado nas seções 4-8 ("S" ou "L")
SEED = 0

import os, sys, json
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

%load_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from IPython.display import Image, display
from src import config as C
from src.common import DEVICE, AMP_DTYPE

print(f"raiz {C.ROOT} | device {DEVICE} ({AMP_DTYPE})")
print(f"não rotuladas {C.N_UNLABELED} | K {C.K_VIEWS} | épocas {C.EPOCHS} (warm-up {C.WARMUP_EPOCHS}) "
      f"| batch {C.BATCH} | {C.IMG_SIZE}px | pool dos probes {C.PROBE_POOL or 'treino inteiro'}")
print(f"alunos {C.SIZES} | seeds {C.SEEDS}")

## 1. Dados e splits fixos

In [ ]:
from src.data import make_splits, load_split
make_splits()          # determinístico: rodar de novo gera exatamente os mesmos arquivos
x_tr, y_tr = load_split("train")
x_te, y_te = load_split("test")
from src.report import CLASSES

fig, ax = plt.subplots(figsize=(8, 3))
w = 0.4
ax.bar(np.arange(9) - w/2, np.bincount(y_tr) / len(y_tr), w, label="train", color="#2a78d6")
ax.bar(np.arange(9) + w/2, np.bincount(y_te) / len(y_te), w, label="test", color="#eb6834")
ax.set_xticks(range(9), CLASSES, rotation=30, ha="right"); ax.set_ylabel("fração"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 6 exemplos por classe (128 px nativos)
rng = np.random.default_rng(0)
fig, axes = plt.subplots(9, 6, figsize=(7, 11))
for c in range(9):
    for j, i in enumerate(rng.choice(np.where(y_tr == c)[0], 6, replace=False)):
        axes[c, j].imshow(x_tr[i]); axes[c, j].axis("off")
    axes[c, 0].set_title(CLASSES[c], fontsize=8, loc="left")
plt.tight_layout(); plt.show()

## 2. Views aumentadas + features do professor (cache)
K views por imagem, geradas uma vez (lightly `VICRegViewTransform` a 128 px). O aluno usa a view
reduzida para 64 px; o professor, a mesma view ampliada para 224 px. Pulado se o cache já existe.

In [ ]:
from src.data import make_views
if not (C.CACHE / "views_meta.json").exists():
    make_views(batch=32, workers=8)
print(json.loads((C.CACHE / "views_meta.json").read_text()))

In [ ]:
# confira se as aumentações fazem sentido: original (128 px) + as K views (64 px)
views = np.load(C.CACHE / f"views_{C.IMG_SIZE}.npy", mmap_mode="r")
unl = np.load(C.SPLITS / "unlabeled_idx.npy")
n_show, K = 5, views.shape[1]
fig, axes = plt.subplots(n_show, K + 1, figsize=(1.6 * (K + 1), 1.7 * n_show))
for r in range(n_show):
    axes[r, 0].imshow(x_tr[unl[r]]); axes[r, 0].set_title(CLASSES[y_tr[unl[r]]], fontsize=7)
    for k in range(K):
        axes[r, k + 1].imshow(views[r, k].transpose(1, 2, 0))
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.show()

## 3. Linha D — professor (checagem de sanidade)
k-NN bem acima do acaso (~11% com 9 classes; a coluna `chance` mostra a classe majoritária),
`emb_std` claramente > 0 e NMI > 0. Se falhar aqui, pare e investigue antes de treinar.

In [ ]:
from src.extract import extract
extract("D")
pd.Series(json.loads((C.RESULTS / "sanity_D.json").read_text()))

## 4. Linha A — aluno com pesos aleatórios (mesma inicialização da seed em B e C)

In [ ]:
extract("A", SIZE, SEED)
pd.Series(json.loads((C.RESULTS / f"sanity_{C.tag('A', SIZE, SEED)}.json").read_text()))

## 5. Linhas B e C — treino
Mesma arquitetura, inicialização, views, épocas, passos, batch, otimizador e schedule; só o objetivo muda.
O progresso aparece abaixo; se interromper (■), rodar a célula de novo retoma da última época.

In [ ]:
def plot_log(tags):
    # uma painel de loss por run (B e C têm escalas diferentes), depois k-NN e std
    fig, axes = plt.subplots(1, len(tags) + 2, figsize=(4.2 * (len(tags) + 2), 3.2))
    for i, t in enumerate(tags):
        p = C.RUNS / t / "log.csv"
        if not p.exists(): continue
        d = pd.read_csv(p)
        axes[i].plot(d.epoch, d.loss, marker=".", color=f"C{i}")
        axes[i].set_title(f"loss {t}", fontsize=10)
        m = d.dropna(subset=["knn_acc"])
        axes[-2].plot(m.epoch, 100 * m.knn_acc, marker="o", color=f"C{i}", label=t)
        axes[-1].plot(m.epoch, m.emb_std_x_sqrt_d, marker="o", color=f"C{i}", label=t)
    axes[-2].set_title("k-NN no conjunto diag (%)", fontsize=10)
    axes[-1].set_title("std por dim × √d  (→0 = colapso)", fontsize=10)
    for a in axes: a.set_xlabel("época"); a.grid(alpha=.3)
    axes[-1].legend(fontsize=8); plt.tight_layout(); plt.show()

from src.train import train
TAG_B, TAG_C = C.tag("B", SIZE, SEED), C.tag("C", SIZE, SEED)

In [ ]:
train("B", SIZE, SEED)
torch.cuda.empty_cache()
plot_log([TAG_B])

In [ ]:
train("C", SIZE, SEED)
torch.cuda.empty_cache()
plot_log([TAG_B, TAG_C])

## 6. Probes (k-NN, SVM linear, SVM RBF) + k-means
C e γ das SVMs são escolhidos **uma vez** em B (100% dos rótulos, CV só no treino) e congelados.

In [ ]:
from src.probe import select, run, HP_PATH
for row, args in [("A", (SIZE, SEED)), ("B", (SIZE, SEED)), ("C", (SIZE, SEED))]:
    extract(row, *args)
if not HP_PATH.exists():
    select(TAG_B)
print(json.loads(HP_PATH.read_text()))
TAGS = [C.tag("A", SIZE, SEED), TAG_B, TAG_C, "D"]
for t in TAGS:
    run(t)

In [ ]:
df = pd.concat([pd.read_csv(C.RESULTS / "probes" / f"{t}.csv") for t in TAGS])
tab = (df.groupby(["probe", "row", "label_frac"]).acc.agg(["mean", "std"]).mul(100).round(1)
         .assign(v=lambda d: d["mean"].astype(str) + " ± " + d["std"].fillna(0).astype(str))
         .v.unstack("label_frac"))
tab.columns = [f"{int(c*100)}%" for c in tab.columns]
display(tab)
cl = pd.DataFrame([json.loads((C.RESULTS / "clusters" / f"{t}.json").read_text()) for t in TAGS])
display(cl[["tag", "feat_dim", "kmeans_nmi", "kmeans_ari"]].round(3))

## 7. Projeção 2-D (UMAP, mesmos parâmetros para os quatro encoders)

In [ ]:
from src.report import fig_projection
fig_projection(SIZE, SEED, "umap")
display(Image(str(C.FIGS / f"projection_{SIZE}.png")))

## 8. Custo: parâmetros, FLOPs, latência, horas de GPU

In [ ]:
from src import cost
cost.main()

## 9. Estudo completo
Roda tudo que falta (principal primeiro, ablação depois), pulando o que já terminou.

In [ ]:
for frac in C.DATA_FRACS:                      # 1.0 (principal) e 0.5 (ablação)
    for size, seeds in C.SEEDS.items():
        for s in seeds:
            for row in ("B", "C"):
                train(row, size, s, frac); torch.cuda.empty_cache()
from src.extract import all_encoders
for j in all_encoders():
    extract(*j)
if not HP_PATH.exists():
    select(C.tag("B", "S", 0))
for p in sorted(C.FEATS.glob("*.npz")):
    run(p.stem)
cost.main()

In [ ]:
# relatório final: tabelas em results/, figuras em figures/
from src.report import run as report
report("umap", "S")
print((C.RESULTS / "main_table.md").read_text())
for f in ["label_efficiency", "gap_vs_params", "ablation_gap", "projection_S"]:
    if (C.FIGS / f"{f}.png").exists():
        display(Image(str(C.FIGS / f"{f}.png")))